In [ ]:
# processes hourly TEMPO files into filtered (filtering done according to TEMPO documentation), daily files containing all hours
# Use this script if you want to process TEMPO data for a different year other than 2024
# if you want an hourly average there is some commented code at the bottom which can be uncommented


In [2]:
#This script processes hourly TEMPO files into filtered, daily files
import matplotlib.pyplot as plt
import netCDF4
import numpy as np
import pandas as pd
import os
import datetime
import glob
import re
import xarray as xr
import dask.array as da


In [8]:
# ================== CONFIG ==================

directory = '/data/daniel/TEMPO/MemphisTempo2023_2025/'  # Folder with TEMPO input files; change to where you downloaded TEMPO data to
prefix = 'TEMPO_NO2_L3_V03_'                # File prefix; no need to modify
output_dir = '/data/daniel/TEMPO/Daily_TEMPO_Memphis_2023_2025/'  # Where to save daily files; change to your data directory
final_output = '/data/daniel/TEMPO/Memphis_hourly_avg_2023_2025/'  # Final output

os.makedirs(output_dir, exist_ok=True)

# ================== DATE RANGE ==================

# Set the range to whatever you want here
start_date = datetime.datetime(2023, 9, 1)  # Change to start date of your data
end_date = datetime.datetime(2025 , 9, 1)  # Change to end date of your data 

# Add 1 day so the last day is included in the range
date_range = pd.date_range(start=start_date, end=end_date)
 

In [8]:
# ================== CONFIG ================== (rsig)

directory = '/home/daniel/TEMPO/'  # Folder with TEMPO input files; change to where you downloaded TEMPO data to
prefix = 'tempo.l2.no2.vertical_column_troposphere_2023-09-01T000000Z_2023-09-07T235959Z.nc'   # File prefix; no need to modify
output_dir = '/data/daniel/TEMPO/rsig_test_download/'  # Where to save daily files; change to your data directory
# final_output = '/data/daniel/TEMPO/Memphis_hourly_avg_2023_2025/'  # Final output

os.makedirs(output_dir, exist_ok=True)

# ================== DATE RANGE ==================

# Set the range to whatever you want here
start_date = datetime.datetime(2023, 9, 1)  # Change to start date of your data
end_date = datetime.datetime(2025 , 9, 1)  # Change to end date of your data 

# Add 1 day so the last day is included in the range
date_range = pd.date_range(start=start_date, end=end_date)
 

In [9]:

# ================== FUNCTIONS ==================

def preprocess_tempo(file_path):
    ds = netCDF4.Dataset(file_path)

    tropVCD = np.squeeze(ds['product/vertical_column_troposphere'][:]) / 1e15
    quality_flag = np.squeeze(ds['product/main_data_quality_flag'][:])
    cloudfrac = np.squeeze(ds['support_data/eff_cloud_fraction'][:])
    sza = np.squeeze(ds['geolocation/solar_zenith_angle'][:])

    lat = ds.variables['latitude'][:]
    lon = ds.variables['longitude'][:]
    lon2d, lat2d = np.meshgrid(lon, lat)

    tropVCD_filled = np.ma.filled(tropVCD, fill_value=np.nan)
    tropVCD_filled[(quality_flag != 0) | (cloudfrac > 0.2) | (sza > 70)] = np.nan

    ds.close()
    return tropVCD_filled, lat2d, lon2d



In [10]:

# ================== DAILY LOOP ==================

for current_day in date_range:
    print(f"Processing {current_day.date()}")

    daily_arrays = []
    date_str = current_day.strftime('%Y%m%d')
    files = glob.glob(os.path.join(directory, prefix + f'{date_str}T*'))

    if not files:
        continue

    for file in files:
        try:
            # Extract timestamp like "20241230T164728" (no 'Z')
            match = re.search(r'(\d{8}T\d{6})', file)
            if not match:
                raise ValueError(f"Timestamp not found in file name: {file}")
            timestamp_str = match.group(1)
            timestamp = pd.to_datetime(timestamp_str, format="%Y%m%dT%H%M%S")

            tropVCD_data, lat2d, lon2d = preprocess_tempo(file)
            tropVCD_data_dask = da.from_array(tropVCD_data, chunks="auto")

            da_xr = xr.DataArray(
                tropVCD_data_dask[None, :, :],
                dims=["time", "lat", "lon"],
                coords={
                    "time": [timestamp],
                    "latitude": (("lat", "lon"), lat2d),
                    "longitude": (("lat", "lon"), lon2d),
                },
                name="NO2"
            )

            daily_arrays.append(da_xr)

        except Exception as e:
            print(f"Skipping file {file} due to error: {e}")

    if daily_arrays:
        print(f"Saving daily file for {current_day.date()}")
        daily_ds = xr.concat(daily_arrays, dim="time")
        daily_ds = daily_ds.sortby("time") 
        daily_ds.to_netcdf(f"{output_dir}/tempo_{date_str}.nc")

print("✅ All daily files written.")



Processing 2023-09-01
Processing 2023-09-02
Processing 2023-09-03
Processing 2023-09-04
Processing 2023-09-05
Processing 2023-09-06
Processing 2023-09-07
Processing 2023-09-08
Processing 2023-09-09
Processing 2023-09-10
Processing 2023-09-11
Processing 2023-09-12
Processing 2023-09-13
Processing 2023-09-14
Processing 2023-09-15
Processing 2023-09-16
Processing 2023-09-17
Processing 2023-09-18
Processing 2023-09-19
Processing 2023-09-20
Processing 2023-09-21
Processing 2023-09-22
Processing 2023-09-23
Processing 2023-09-24
Processing 2023-09-25
Processing 2023-09-26
Processing 2023-09-27
Processing 2023-09-28
Processing 2023-09-29
Processing 2023-09-30
Processing 2023-10-01
Processing 2023-10-02
Processing 2023-10-03
Processing 2023-10-04
Processing 2023-10-05
Processing 2023-10-06
Processing 2023-10-07
Processing 2023-10-08
Processing 2023-10-09
Processing 2023-10-10
Processing 2023-10-11
Processing 2023-10-12
Processing 2023-10-13
Processing 2023-10-14
Processing 2023-10-15
Processing

# ================== HOURLY AVERAGE (Optional) ==================


print("Opening all daily files (lazy)...")
ds = xr.open_mfdataset(f"{output_dir}/tempo_*.nc", combine='by_coords', chunks={})

print("Computing hourly average...")
ds['hour'] = ds['time'].dt.hour
tempo_hourly_avg = ds.groupby("hour").mean(dim="time")

print("Saving final hourly average...")
tempo_hourly_avg.to_netcdf(final_output)

print(f"✅ Saved hourly average to: {final_output}")
